In [1]:
#import new dataframe + cleaning script
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree

In [2]:
temp_df = pd.read_csv('data/caml_satellite_matchup.csv')

In [3]:
temp_df.head(5)

,system:index,NIR,SWIR,blue,cyanobacteria_abundance,green,longitude,latitude,ndci,ndti,ndvi,ndwi,red,red_edge,sample_date,satellite_date,sensor,temp_c,.geo
0,00000000000000000c88,0.195500,0.27050,0.109000,6.061,0.152500,-95.479569,39.141931,0.106331,-0.090367,0.033571,-0.123563,0.182800,0.2263,2020-06-29,2020-06-28,s2,29.543909,"{""type"":""Point"",""coordinates"":[-95.47956935478..."
1,000000000000000011e3,-999.000000,-999.00000,-999.000000,0.945,-999.000000,-96.904618,39.094473,-999.000000,-999.000000,-999.000000,-999.000000,-999.000000,-999.0000,2014-10-06,NONE,NONE,18.222437,"{""type"":""Point"",""coordinates"":[-96.90461778765..."
2,0000000000000000125c,0.046300,0.05590,0.032400,1.575,0.041600,-96.904618,39.094473,0.267211,0.139726,0.191763,-0.053470,0.031400,0.0543,2017-08-28,2017-08-23,s2,26.154230,"{""type"":""Point"",""coordinates"":[-96.90461778765..."
3,000000000000000012d3,0.089162,0.05784,0.028965,9.135,0.065375,-96.904618,39.094473,-999.000000,0.134318,0.282406,-0.153927,0.049892,-999.0000,2015-08-10,2015-08-12,l8,33.776196,"{""type"":""Point"",""coordinates"":[-96.90461778765..."
4,000000000000000019c9,-999.000000,-999.00000,-999.000000,5.355,-999.000000,-96.904618,39.094473,-999.000000,-999.000000,-999.000000,-999.000000,-999.000000,-999.0000,2014-09-15,NONE,NONE,21.517633,"{""type"":""Point"",""coordinates"":[-96.90461778765..."


In [4]:
#cleaning script
def thin_points(date_group: pd.DataFrame, area: int) -> pd.DataFrame:
    """
    inspect groups of rows that come from the same date but different locations.
    if locations are within [area] meters of each other, keep only one row in that area.
    """
    # if the group only contains 1 sample, do nothing
    if len(date_group) < 2:
        return date_group
    
    temp_group = date_group.reset_index(drop=True)
        
    # convert latitude and longitude to radians
    coords = np.deg2rad(temp_group[["latitude", "longitude"]].values)

    # build a ball tree using haversine metric for distance (accounts for spherical earth)
    tree = BallTree(coords, metric="haversine")
    radians_area = area / 6371000.0

    # find neighboring points for each point in the group
    indices = tree.query_radius(coords, r=radians_area)

    already_covered = set()
    keep_indices = []
    
    for i, neighbors in enumerate(indices):     # loop through each point in the group
        if i not in already_covered:            # if the point has not been covered:
            keep_indices.append(i)                  # keep it to represent the neighbor group
            already_covered.update(neighbors)       # add all of its neighbors to already_covered
    
    return temp_group.iloc[keep_indices].copy()

def load_clean_data(path: str) -> pd.DataFrame:

    # load data
    try:
        df = pd.read_csv(path)
    except Exception as e:
        print(f"data loading unsuccessful: {e}")
        return

    print("rows pre-cleaning:", df.shape[0])

    # remove rows where sensor is NONE
    print("rows removed due to no sensor match:", df[df["sensor"]=="NONE"].shape[0])
    df = df[df["sensor"]!="NONE"]

    # remove rows with impossible band values (not including red edge since all l8 values will be -999)
    band_cols = ["blue", "green", "red", "NIR", "SWIR"]
    valid_bands = ((df[band_cols] > 0) & (df[band_cols] < 1.2)).all(axis=1)
    print("rows removed due to impossible band values:", df[~valid_bands].shape[0])
    df = df[valid_bands].copy()

    # remove columns system:index and .geo
    df.drop(columns=["system:index", ".geo"], inplace=True)

    # thin data by keeping 1 point for each 50m area on the same day
    rows_before_thinning = df.shape[0]
    thinned_groups = []
    for _, group in df.groupby('satellite_date', sort=False):  # group by date
        thinned = thin_points(group, 50)                       # thin each group
        thinned_groups.append(thinned)                         # add to thinned groups list
    df = pd.concat(thinned_groups, ignore_index=True)          # turn back into df
    print("rows removed due to spatial overlap:", rows_before_thinning - df.shape[0])

    # replace missing values with nans
    df.replace(-999, np.nan, inplace=True)

    # scale abundance values
    df['log_abundance'] = np.log10(df['cyanobacteria_abundance'] + 1)

    # turn dates into datetime objects
    df["sample_date"] = pd.to_datetime(df["sample_date"], format="%Y-%m-%d", yearfirst=True)
    df["satellite_date"] = pd.to_datetime(df["satellite_date"], format="%Y-%m-%d", yearfirst=True)

    # create column for difference in days between sample and sensor measurement
    df["days_difference"] = (df["sample_date"] - df["satellite_date"]).dt.days.abs()

    # encode sensor column and remove string sensor column
    df["is_s2"] = (df["sensor"]=="s2").astype(int)
    df.drop(columns="sensor", inplace=True)

    print("rows post-cleaning:", df.shape[0])
    return df

In [5]:
#run cleaning script on existing dataframe
df = load_clean_data('data/caml_satellite_matchup.csv')

rows pre-cleaning: 23570
rows removed due to no sensor match: 10091
rows removed due to impossible band values: 332
rows removed due to spatial overlap: 3967
rows post-cleaning: 9180


In [6]:
df.head(5)

,NIR,SWIR,blue,cyanobacteria_abundance,green,longitude,latitude,ndci,ndti,ndvi,ndwi,red,red_edge,sample_date,satellite_date,temp_c,log_abundance,days_difference,is_s2
0,0.1955,0.2705,0.1090,6.061000,0.1525,-95.479569,39.141931,0.106331,-0.090367,0.033571,-0.123563,0.1828,0.2263,2020-06-29,2020-06-28,29.543909,0.848866,1,1
1,0.1694,0.2252,0.1043,17.993000,0.1360,-95.642822,38.818080,0.303571,0.012658,0.121854,-0.109365,0.1326,0.2482,2020-06-30,2020-06-28,31.308298,1.278594,2,1
2,0.1701,0.1458,0.1228,188.654007,0.1443,-95.521146,39.686240,0.163079,0.008386,0.090385,-0.082061,0.1419,0.1972,2020-06-29,2020-06-28,29.750940,2.277962,1,1
3,0.0463,0.0559,0.0324,1.575000,0.0416,-96.904618,39.094473,0.267211,0.139726,0.191763,-0.053470,0.0314,0.0543,2017-08-28,2017-08-23,26.154230,0.410777,5,1
4,0.1068,0.1121,0.0700,5.040000,0.0844,-96.905898,39.077751,0.142266,0.008966,0.125988,-0.117155,0.0829,0.1104,2017-08-21,2017-08-23,29.083255,0.781037,2,1


In [7]:
s2_df = df[df['is_s2'] == 1]

In [8]:
s2_df.shape
#EVENTUALLY look at distribution of these points across U.S. --> see what effects this has

(6405, 19)

In [9]:
#use s2_df for building time series dataframe
#accessing google earth engine
import ee
ee.Initialize(project = 'algaebloomforecaster')
s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')

In [93]:
#create a smaller, temp_df
temp_df = df.iloc[:5]
temp_df.head(5)

,NIR,SWIR,blue,cyanobacteria_abundance,green,latitude,longitude,ndci,ndti,ndvi,ndwi,red,red_edge,sample_date,satellite_date,temp_c,log_abundance,days_difference,is_s2
0,0.1955,0.2705,0.1090,6.061000,0.1525,-95.479569,39.141931,0.106331,-0.090367,0.033571,-0.123563,0.1828,0.2263,2020-06-29,2020-06-28,29.543909,0.848866,1,1
1,0.1694,0.2252,0.1043,17.993000,0.1360,-95.642822,38.818080,0.303571,0.012658,0.121854,-0.109365,0.1326,0.2482,2020-06-30,2020-06-28,31.308298,1.278594,2,1
2,0.1701,0.1458,0.1228,188.654007,0.1443,-95.521146,39.686240,0.163079,0.008386,0.090385,-0.082061,0.1419,0.1972,2020-06-29,2020-06-28,29.750940,2.277962,1,1
3,0.0463,0.0559,0.0324,1.575000,0.0416,-96.904618,39.094473,0.267211,0.139726,0.191763,-0.053470,0.0314,0.0543,2017-08-28,2017-08-23,26.154230,0.410777,5,1
4,0.1068,0.1121,0.0700,5.040000,0.0844,-96.905898,39.077751,0.142266,0.008966,0.125988,-0.117155,0.0829,0.1104,2017-08-21,2017-08-23,29.083255,0.781037,2,1


In [100]:
ee.Initialize(project="algaebloomforecaster")
s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")

def scale_bands(image):
    """Scale S2 optical bands and compute indices."""
    image = ee.Image(image)  # ✅ explicit cast fixes the Element error
    optical = image.select(
        ["B2", "B3", "B4", "B5", "B8A", "B11"],
        ["blue", "green", "red", "NIR", "red_edge", "SWIR"]
    ).multiply(0.0001)

    ndci = optical.normalizedDifference(["red_edge", "red"]).rename("ndci")
    ndvi = optical.normalizedDifference(["NIR", "red"]).rename("ndvi")
    ndwi = optical.normalizedDifference(["green", "NIR"]).rename("ndwi")
    ndti = optical.normalizedDifference(["green", "red"]).rename("ndti")

    return optical.addBands([ndci, ndvi, ndwi, ndti]) \
                  .copyProperties(image, ["system:time_start", "CLOUDY_PIXEL_PERCENTAGE"])

def extract_satellite_data(feature):
    raw_date = ee.Number(feature.get("date")).format('%.0f')
    sample_date = ee.Date.parse("YYYYMMdd", raw_date)

    geom = feature.geometry()
    coords = geom.coordinates()
    lon = coords.get(0)
    lat = coords.get(1)
    region = geom.buffer(30)

    start = sample_date.advance(-90, 'day')
    end = sample_date

    s2_collection = (
        s2.filterBounds(geom)
          .filterDate(start, end)
          .map(scale_bands)
    )

    def process_image(img):
        img = ee.Image(img)  # ✅ explicit cast

        # ✅ Cloud score from image metadata — no QA60 needed
        cloud_score = ee.Number(img.get('CLOUDY_PIXEL_PERCENTAGE')).divide(100)

        stats = img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=region,
            scale=10,
            maxPixels=1e9
        )

        sat_date = ee.Algorithms.If(
            img.get('system:time_start'),
            ee.Date(img.get('system:time_start')).format('YYYY-MM-dd'),
            'NONE'
        )

        defaults = ee.Dictionary({
            "blue": -999, "green": -999, "red": -999, "NIR": -999,
            "red_edge": -999, "SWIR": -999, "ndci": -999, "ndvi": -999,
            "ndwi": -999, "ndti": -999
        })
        props = defaults.combine(stats, True)

        return ee.Feature(None, props) \
            .set('satellite_date', sat_date) \
            .set('cloud_score', cloud_score) \
            .set('sample_date', sample_date.format("YYYY-MM-dd")) \
            .set('latitude', lat) \
            .set('longitude', lon) \
            .set('cyanobacteria_abundance', feature.get('abun'))

    return s2_collection.map(process_image)

def df_to_fc(df):
    """Convert pandas DataFrame to GEE FeatureCollection."""
    features = []
    for idx, row in df.iterrows():
        point = ee.Geometry.Point([row['latitude'], row['longitude']])
        feature = ee.Feature(point, {
            'date': int(row['sample_date'].strftime('%Y%m%d')),
            'abun': float(row['cyanobacteria_abundance'])
        })
        features.append(feature)
    return ee.FeatureCollection(features)


fc = df_to_fc(df)

#more testing
test_feature = ee.Feature(fc.first())
raw_date = ee.Number(test_feature.get("date")).format('%.0f')
sample_date = ee.Date.parse("YYYYMMdd", raw_date)
geom = test_feature.geometry()
start = sample_date.advance(-90, 'day')
end = sample_date
test_collection = s2.filterBounds(geom).filterDate(start, end)
processed_fc = fc.map(extract_satellite_data).flatten()


export_cols = [
    "latitude", "longitude", "cyanobacteria_abundance",
    "sample_date", "satellite_date", "cloud_score",
    "blue", "green", "red", "red_edge", "NIR", "SWIR",
    "ndci", "ndvi", "ndwi", "ndti"
]

export = processed_fc.select(export_cols)

task = ee.batch.Export.table.toDrive(
    collection=export,
    description="caml_90days_previous",
    fileFormat="CSV"
)
task.start()
print(task.status())

{'state': 'READY', 'description': 'caml_90days_previous', 'priority': 100, 'creation_timestamp_ms': 1776115956559, 'update_timestamp_ms': 1776115956559, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'QZXKHT6RO7EYOEW3IPL5Z2W7', 'name': 'projects/algaebloomforecaster/operations/QZXKHT6RO7EYOEW3IPL5Z2W7'}


In [11]:
temp_df = df.iloc[:5]
temp_df.head(5)

,NIR,SWIR,blue,cyanobacteria_abundance,green,longitude,latitude,ndci,ndti,ndvi,ndwi,red,red_edge,sample_date,satellite_date,temp_c,log_abundance,days_difference,is_s2
0,0.1955,0.2705,0.1090,6.061000,0.1525,-95.479569,39.141931,0.106331,-0.090367,0.033571,-0.123563,0.1828,0.2263,2020-06-29,2020-06-28,29.543909,0.848866,1,1
1,0.1694,0.2252,0.1043,17.993000,0.1360,-95.642822,38.818080,0.303571,0.012658,0.121854,-0.109365,0.1326,0.2482,2020-06-30,2020-06-28,31.308298,1.278594,2,1
2,0.1701,0.1458,0.1228,188.654007,0.1443,-95.521146,39.686240,0.163079,0.008386,0.090385,-0.082061,0.1419,0.1972,2020-06-29,2020-06-28,29.750940,2.277962,1,1
3,0.0463,0.0559,0.0324,1.575000,0.0416,-96.904618,39.094473,0.267211,0.139726,0.191763,-0.053470,0.0314,0.0543,2017-08-28,2017-08-23,26.154230,0.410777,5,1
4,0.1068,0.1121,0.0700,5.040000,0.0844,-96.905898,39.077751,0.142266,0.008966,0.125988,-0.117155,0.0829,0.1104,2017-08-21,2017-08-23,29.083255,0.781037,2,1


In [16]:
#now creating the set of time series data after the sample date

ee.Initialize(project="algaebloomforecaster")
s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")

def scale_bands(image):
    """Scale S2 optical bands and compute indices."""
    image = ee.Image(image)  # ✅ explicit cast fixes the Element error
    optical = image.select(
        ["B2", "B3", "B4", "B5", "B8A", "B11"],
        ["blue", "green", "red", "NIR", "red_edge", "SWIR"]
    ).multiply(0.0001)

    ndci = optical.normalizedDifference(["red_edge", "red"]).rename("ndci")
    ndvi = optical.normalizedDifference(["NIR", "red"]).rename("ndvi")
    ndwi = optical.normalizedDifference(["green", "NIR"]).rename("ndwi")
    ndti = optical.normalizedDifference(["green", "red"]).rename("ndti")

    return optical.addBands([ndci, ndvi, ndwi, ndti]) \
                  .copyProperties(image, ["system:time_start", "CLOUDY_PIXEL_PERCENTAGE"])

def extract_satellite_data(feature):
    raw_date = ee.Number(feature.get("date")).format('%.0f')
    sample_date = ee.Date.parse("YYYYMMdd", raw_date)

    geom = feature.geometry()
    coords = geom.coordinates()
    lon = coords.get(0)
    lat = coords.get(1)
    region = geom.buffer(30)

    start = sample_date
    end = sample_date.advance(60, 'day')

    s2_collection = (
        s2.filterBounds(geom)
          .filterDate(start, end)
          .map(scale_bands)
    )

    def process_image(img):
        img = ee.Image(img)  # ✅ explicit cast

        # ✅ Cloud score from image metadata — no QA60 needed
        cloud_score = ee.Number(img.get('CLOUDY_PIXEL_PERCENTAGE')).divide(100)

        stats = img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=region,
            scale=10,
            maxPixels=1e9
        )

        sat_date = ee.Algorithms.If(
            img.get('system:time_start'),
            ee.Date(img.get('system:time_start')).format('YYYY-MM-dd'),
            'NONE'
        )

        defaults = ee.Dictionary({
            "blue": -999, "green": -999, "red": -999, "NIR": -999,
            "red_edge": -999, "SWIR": -999, "ndci": -999, "ndvi": -999,
            "ndwi": -999, "ndti": -999
        })
        props = defaults.combine(stats, True)

        return ee.Feature(None, props) \
            .set('satellite_date', sat_date) \
            .set('cloud_score', cloud_score) \
            .set('sample_date', sample_date.format("YYYY-MM-dd")) \
            .set('latitude', lat) \
            .set('longitude', lon) \
            .set('cyanobacteria_abundance', feature.get('abun'))

    return s2_collection.map(process_image)

def df_to_fc(df):
    """Convert pandas DataFrame to GEE FeatureCollection."""
    features = []
    for idx, row in df.iterrows():
        point = ee.Geometry.Point([row['longitude'], row['latitude']])
        feature = ee.Feature(point, {
            'date': int(row['sample_date'].strftime('%Y%m%d')),
            'abun': float(row['cyanobacteria_abundance'])
        })
        features.append(feature)
    return ee.FeatureCollection(features)


fc = df_to_fc(df)

#more testing
test_feature = ee.Feature(fc.first())
raw_date = ee.Number(test_feature.get("date")).format('%.0f')
sample_date = ee.Date.parse("YYYYMMdd", raw_date)
geom = test_feature.geometry()
start = sample_date
end = sample_date.advance(60, 'day')
test_collection = s2.filterBounds(geom).filterDate(start, end)
processed_fc = fc.map(extract_satellite_data).flatten()


export_cols = [
    "latitude", "longitude", "cyanobacteria_abundance",
    "sample_date", "satellite_date", "cloud_score",
    "blue", "green", "red", "red_edge", "NIR", "SWIR",
    "ndci", "ndvi", "ndwi", "ndti"
]

export = processed_fc.select(export_cols)

task = ee.batch.Export.table.toDrive(
    collection=export,
    description="caml_60days_post",
    fileFormat="CSV"
)
task.start()
print(task.status())

{'state': 'READY', 'description': 'caml_60days_post', 'priority': 100, 'creation_timestamp_ms': 1776633862654, 'update_timestamp_ms': 1776633862654, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'YXED7FGIMCUX5LBDWB7SSQF2', 'name': 'projects/algaebloomforecaster/operations/YXED7FGIMCUX5LBDWB7SSQF2'}


In [15]:
task.cancel()